## This code is for  segmenting the macular images

### Import necessary library files

In [1]:
import os
from os import environ
environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
environ['CUDA_VISIBLE_DEVICES'] = '1' #Select the GPU Card
print('Process id: ',os.getpid())


from os.path import dirname, abspath
from os import listdir
import os, os.path, time
import time as time
import timeit

from sys import argv
import sys



########### Load Keras Modules Needed ==========================================
import tensorflow as tf
import tensorflow.keras
# import tensorflow.keras.backend as K

from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, load_model, Model

from tensorflow.keras.layers import Input, GlobalAveragePooling2D, GlobalMaxPooling2D, \
                                    concatenate, average, Add, Reshape, Permute, Conv2D, \
                                    Conv2DTranspose, SeparableConv2D,PReLU, ELU, \
                                    BatchNormalization, \
                                    MaxPooling2D, ZeroPadding2D, UpSampling2D, \
                                    AveragePooling2D,Cropping2D, \
                                    Flatten, Dense, Dropout, Activation, Reshape,\
                                    SpatialDropout2D,Average, Multiply, Lambda, Resizing, Concatenate


from tensorflow.keras.optimizers import RMSprop, SGD, Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping,\
                            LearningRateScheduler, Callback
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model,to_categorical,Sequence



# Load other functions ==================================================
import glob

import numpy as np
from numpy import zeros, newaxis
import h5py, pickle
from random import randint, choice, shuffle, sample
from itertools import product

from ipywidgets import interact, interactive, fixed #interact_manualten
import ipywidgets as widgets
from scipy import signal

import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import shutil


Process id:  857962


2025-07-21 10:45:41.726897: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-21 10:45:41.738448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753094741.752827  857962 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753094741.757155  857962 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-21 10:45:41.773612: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### testname

In [2]:
testname="Attention_UNet_cce_pf_1"

# collect images

In [3]:
imgpath = "./../../../../../mnt/DATA/sda1/ritu/Data/"

OCT_list = sorted(glob.glob(imgpath+'OCT/*.npy'))
label_list = sorted(glob.glob(imgpath+'Label/*.npy'))
print(len(OCT_list))
print(len(label_list))

print(OCT_list[0])
print(label_list[0])


5682
5682
./../../../../../mnt/DATA/sda1/ritu/Data/OCT/Spec_G_Aravind_101_55658.npy
./../../../../../mnt/DATA/sda1/ritu/Data/Label/Spec_G_Aravind_101_55658_label.npy


In [4]:
OCT_img = []
Label_img = []

### list of new images
for i in range(len(OCT_list)):
    img=np.load(OCT_list[i]) #read the image
    img = np.asarray(img, dtype=np.float32)
    img=(img-np.min(img))/(np.max(img)-np.min(img)) #normalize the intensities to bring between 0 and 1
    OCT_img.append(img)

    temp_name = (OCT_list[i].split('/')[-1]).split('.')[0]+'_label.npy'
    l_img=np.load(imgpath+'Label/' + temp_name)
    Label_img.append(l_img)


OCT_img = np.asarray(OCT_img)
Label_img = np.asarray(Label_img)

print("number of new images and labels: ",np.shape(OCT_img), np.shape(Label_img))

number of new images and labels:  (5682, 200, 400) (5682, 200, 400)


In [5]:
print(np.unique(Label_img[0]))

[0 1 2 3 4 5 6 7]


In [6]:
def process_label(label_file):
        """
        This fucntion takes label files with layers marked from 1 to 8, not 0 to 7
        """
        colors = np.zeros((9,3),dtype=np.uint8)

        #### Define colors for each label
        colors[0] = [255, 255, 255]  # Background (white)
        colors[1] = [255, 0, 0]      # Class 1 (Red)
        colors[2] = [0, 255, 0]      # Class 2 (Green)
        colors[3] = [0, 0, 255]      # Class 3 (Blue)
        colors[4] = [75, 0, 130]     # Class 4 (Indigo)
        colors[5] = [255, 165, 0]    # Class 5 (Orange)
        colors[6] = [255, 255, 0]    # Class 6 (Yellow)      
        colors[7] = [238, 130, 238]  # Class 7 (Violet)

        H = np.shape(label_file)[0]
        W = np.shape(label_file)[1]

        img = np.zeros((H,W,3),dtype=np.uint8)
        for i,j in product(range(H),range(W)):
            img[i, j] = colors[label_file[i,j]-1]

        return np.asarray(img)

In [7]:
@interact(x=(0,len(OCT_img)-1,1)) #Function responsible for creating the slider. Usage format @interact(start,end,step size)
def h(x=0): #Starting number of the slider
#     print(OCT_list[x].split('/')[-1])
    f = plt.figure(figsize=(12,6))

    f.add_subplot(1,3, 1)
    plt.imshow(OCT_img[x], cmap='gray')

    f.add_subplot(1,3, 2)
    color_label= process_label(Label_img[x]+1)
    plt.imshow(color_label)

    f.add_subplot(1,3, 3)
    plt.imshow(OCT_img[x], cmap="gray")
    plt.imshow(color_label,alpha=0.25)
    plt.show()

interactive(children=(IntSlider(value=0, description='x', max=5681), Output()), _dom_classes=('widget-interact…

# shuffle images

In [8]:
### shuffle new images
zipimgs=list(zip(OCT_img,Label_img)) #zip the labels  and image data
shuffle(zipimgs) #shuffle the data
OCT_img1,Label_img1=zip(*zipimgs)

print("number of new images and labels after resize: ",np.shape(OCT_img1), np.shape(Label_img1))

Label_img_exp = to_categorical(Label_img1)
print("number of new images and labels after exp: ",np.shape(OCT_img1), np.shape(Label_img_exp))


number of new images and labels after resize:  (5682, 200, 400) (5682, 200, 400)
number of new images and labels after exp:  (5682, 200, 400) (5682, 200, 400, 8)


In [9]:
@interact(x=(0,len(OCT_img1)-1,1)) #Function responsible for creating the slider. Usage format @interact(start,end,step size)
def h(x=0): #Starting number of the slider
#     print(OCT_list[x].split('/')[-1])
    f = plt.figure(figsize=(12,6))

    f.add_subplot(1,3, 1)
    plt.imshow(OCT_img1[x], cmap='gray')

    f.add_subplot(1,3, 2)
    color_label= process_label(np.argmax(Label_img_exp[x], axis=2)+1)
    plt.imshow(color_label)

    f.add_subplot(1,3, 3)
    plt.imshow(OCT_img1[x], cmap="gray")
    plt.imshow(color_label,alpha=0.2)
    plt.show()

interactive(children=(IntSlider(value=0, description='x', max=5681), Output()), _dom_classes=('widget-interact…

# data split

In [10]:
no_train_images=300 # Number of training images
no_test_images=100#Number of testing images
no_val_images=100 # Number of images in the validation set


W, H = 400,200 #dimension of each image from the cirrus device
nb_classes=8  #11 Layers

In [11]:
train_imgs  =OCT_img1[0:300]
train_labels=Label_img_exp[0:300]

val_imgs  =OCT_img1[300:400]
val_labels=Label_img_exp[300:400]

test_imgs  =OCT_img1[400:500]
test_labels=Label_img_exp[400:500]

train_images=np.expand_dims(train_imgs,axis=3)
val_images=np.expand_dims(val_imgs,axis=3)
test_images=np.expand_dims(test_imgs,axis=3)

print('\nTraining data dimension: {} for labels and {} for images'.format(np.shape(train_labels),np.shape(train_images)))
print('\nValidation data dimension: {} for labels and {} for images'.format(np.shape(val_labels),np.shape(val_images)))
print('\nTesting data dimension: {} for labels and {} for images'.format(np.shape(test_labels),np.shape(test_images)))


Training data dimension: (300, 200, 400, 8) for labels and (300, 200, 400, 1) for images

Validation data dimension: (100, 200, 400, 8) for labels and (100, 200, 400, 1) for images

Testing data dimension: (100, 200, 400, 8) for labels and (100, 200, 400, 1) for images


# Data generation function

In [12]:
def DataGen(X, y, batch_sz,mode):
    alpha, alpha2, sigma = 10, 15, 50
    #alpha, alpha2, sigma = 0, 0, 60

    c = list(zip(X, y))
    croph, cropw = H,W

    shape = X[0].shape[:2]
    x_mesh, y_mesh = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))

    Xpatches = np.ndarray((batch_sz, croph, cropw, 1), dtype=np.float32)
    ypatches = np.ndarray((batch_sz, croph, cropw, nb_classes), dtype=np.float32)

    #create a mask for later purposes
    black_dx = 60
    black_dy = 20
    dark_mask = np.zeros((black_dx, black_dy))
    for k in range(black_dy):
        dark_mask[:,k] = (np.abs(k-black_dy//2) / (black_dy/2.))**2

    image_index = 0
    batch_index = 0
    while (mode==True):
        #if the batch is full, release the batch
        if batch_index == batch_sz:
            batch_index = 0
            yield Xpatches, ypatches

        if image_index >= len(X):
            image_index = 0

        #do the following at the start of each epoch
        if image_index == 0:
            #shuffle the data
            shuffle(c)
            X, y = zip(*c)
            #below is used once per epoch for the elastic deformation
            g_1d = signal.windows.gaussian(300, sigma)
            kernel_deform = np.outer(g_1d, g_1d)
            dx = signal.fftconvolve(np.random.rand(*shape) * 2 - 1, kernel_deform, mode='same')
            dy = signal.fftconvolve(np.random.rand(*shape) * 2 - 1, kernel_deform, mode='same')
            dx = alpha * (dx - np.mean(dx)) / np.std(dx)
            dy = alpha2 * (dy - np.mean(dy))/ np.std(dy)
            indices_x, indices_y = x_mesh+dx, y_mesh+dy
            indices_x_clipped = np.clip(indices_x, a_min=0, a_max=shape[1]-1)
            indices_y_clipped = np.clip(indices_y, a_min=0, a_max=shape[0]-1)


        #img_height, img_width = X[image_index].shape[:-1]
        ax, ay = np.copy(X[image_index]), np.copy(y[image_index])

        #flip with probability 1/2
        if randint(0, 1):
            ax = ax[:, ::-1, :]
            ay = ay[:, ::-1, :]

        #Intensity nonlinear shift
        if True:
            p = np.random.uniform(low=0.6, high=1.4)
            ax[:,:,0] = ax[:,:,0]**p
            a = np.random.uniform(low=0, high=0.1)
            b = np.random.uniform(low=0, high=0.1)
            ax[:,:,0] = -a + (1+a+b) * ax[:,:,0]
            ax = np.clip(ax, 0., 1.)

        #add black boxes
        if True:
            for k in range(20):
                black_dx = 60
                black_dy = 20
                black_x, black_y = randint(0, ax.shape[0] -  black_dx), randint(0, ax.shape[1] -  black_dy)
                #ax[black_x:(black_x+black_dx), black_y:(black_y+black_dy), 0] = 0
                intensity_dark = np.random.uniform(low=0.2, high=0.8)
                window_to_darken = ax[black_x:(black_x+black_dx), black_y:(black_y+black_dy), 0]
                ax[black_x:(black_x+black_dx), black_y:(black_y+black_dy), 0] = window_to_darken * (intensity_dark + (1.-intensity_dark)*dark_mask)

        #elastic deformation
        if True:
            ax[:,:,0] = ax[indices_y_clipped.astype(int),indices_x_clipped.astype(int),0]
            for k in range(nb_classes):
                ay[:,:,k] = ay[indices_y_clipped.astype(int),indices_x_clipped.astype(int),k]

        #add Gaussian **multiplicative** noise with variable intensity
        if True:
            intensity_noise = np.random.uniform(low=0, high=0.1)
            ax[:,:,0] = ax[:,:,0]*(1+ intensity_noise*np.random.normal(loc=0, scale=1, size=shape[0]*shape[1]).reshape(shape[0],shape[1]))

        #add Gaussian **additive** noise with variable intensity
        if True:
            intensity_noise = np.random.uniform(low=0, high=0.1)
            ax[:,:,0] = ax[:,:,0]  + intensity_noise*np.random.normal(loc=0, scale=1, size=shape[0]*shape[1]).reshape(shape[0],shape[1])


        if False:
            #Define the noise and avg parameters
            gaussian_noise=0
            white_noise=0
            white_mu=8.0963e-06
            sum_averaging=0
            averaging=1 # no averaging is needed. Averaging is shitty.
            random.seed(1)
            for avg in xrange(averaging):
                speckle_noise=-math.log10(np.random.randint(1,100))
                white_noise=-white_mu*math.log10(np.random.randint(1,100))
                sum_averaging=sum_averaging+speckle_noise*ax[:,:,0]+white_noise

            ax[:,:,0]=1/(averaging)*sum_averaging

        #make sure that pixel intensity is in [0,1] and labels are integers
        ax = np.clip(ax, 0., 1.)
        ay = np.clip(ay, 0., 1.)
        ay = np.rint(ay)

        Xpatches[batch_index] = ax
        ypatches[batch_index] = ay

        batch_index = batch_index + 1
        image_index = image_index + 1

In [13]:
# Generate augmentation
batch_sz = 20
training_gen = DataGen(train_images, train_labels, batch_sz,mode=True)
val_gen = DataGen(val_images, val_labels, batch_sz,mode=False)


im,seg = next(training_gen)
print("Dim(image)=",np.shape(im))
print("Dim(Label)=",np.shape(seg))

Dim(image)= (20, 200, 400, 1)
Dim(Label)= (20, 200, 400, 8)


In [14]:
@interact(x=(0,len(im)-1,1)) #Function responsible for creating the slider. Usage format @interact(start,end,step size)
def h(x=0): #Starting number of the slider

    f = plt.figure(figsize=(12,6))

    f.add_subplot(1,3, 1)
    plt.imshow(im[x,:,:,0], cmap='gray')

    f.add_subplot(1,3, 2)
    color_label= process_label(np.argmax(seg[x], axis=2)+1)
    plt.imshow(color_label)

    f.add_subplot(1,3, 3)
    plt.imshow(im[x,:,:,0], cmap="gray")
    plt.imshow(color_label,alpha=0.2)
    plt.show()

interactive(children=(IntSlider(value=0, description='x', max=19), Output()), _dom_classes=('widget-interact',…

 # UNet architecture

In [15]:
# Convolutional block (2 convs)
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

# Attention gate with static resizing
def attention_gate(x, g, inter_channels):
    x_shape = K.int_shape(x)
    g = layers.Resizing(x_shape[1], x_shape[2])(g)

    theta_x = layers.Conv2D(inter_channels, 1, padding='same')(x)
    phi_g = layers.Conv2D(inter_channels, 1, padding='same')(g)

    add = layers.Add()([theta_x, phi_g])
    relu = layers.Activation('relu')(add)
    psi = layers.Conv2D(1, 1, padding='same')(relu)
    sigmoid = layers.Activation('sigmoid')(psi)

    return layers.Multiply()([x, sigmoid])

# Resize decoder output to match encoder skip connection
def resize_like(source, target):
    target_shape = K.int_shape(target)
    return layers.Resizing(target_shape[1], target_shape[2])(source)

# Main Attention UNet function
def attention_unet(img_rows, img_cols, num_classes=1):
    inputs = tf.keras.Input(shape=(img_rows, img_cols, 1))

    # Encoder
    c1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 256)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 512)
    p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 1024)

    # Decoder with attention and resizing
    u4 = layers.UpSampling2D(interpolation='bilinear')(bn)
    att4 = attention_gate(c4, u4, 512)
    u4 = resize_like(u4, att4)
    m4 = layers.Concatenate()([u4, att4])
    d4 = conv_block(m4, 512)

    u3 = layers.UpSampling2D(interpolation='bilinear')(d4)
    att3 = attention_gate(c3, u3, 256)
    u3 = resize_like(u3, att3)
    m3 = layers.Concatenate()([u3, att3])
    d3 = conv_block(m3, 256)

    u2 = layers.UpSampling2D(interpolation='bilinear')(d3)
    att2 = attention_gate(c2, u2, 128)
    u2 = resize_like(u2, att2)
    m2 = layers.Concatenate()([u2, att2])
    d2 = conv_block(m2, 128)

    u1 = layers.UpSampling2D(interpolation='bilinear')(d2)
    att1 = attention_gate(c1, u1, 64)
    u1 = resize_like(u1, att1)
    m1 = layers.Concatenate()([u1, att1])
    d1 = conv_block(m1, 64)

    # Output layer
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(d1)

    model = Model(inputs, outputs)
    return model

In [16]:
# def attention_gate(x, g, filters):
#     """ Attention Gate: Enhances relevant features """
#     # Apply 'same' padding before the Conv2D layers
#     x_padded = tf.keras.layers.ZeroPadding2D(padding=((1, 1), (1, 1)))(x)  # Pad x
#     g_padded = tf.keras.layers.ZeroPadding2D(padding=((1, 1), (1, 1)))(g)  # Pad g


#     theta_x = Conv2D(filters, kernel_size=1, padding='same')(x_padded)
#     phi_g = Conv2D(filters, kernel_size=1, padding='same')(g_padded)
#     phi_g = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (theta_x.shape[1], theta_x.shape[2])))(phi_g)

#     add = Add()([theta_x, phi_g])
#     act = Activation('relu')(add)
#     psi = Conv2D(1, kernel_size=1, padding='same', activation='sigmoid')(act)

#     x_shape = tf.keras.backend.int_shape(x)
#     target_height = x_shape[1]
#     target_width = x_shape[2]

#     psi_up = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (target_height, target_width)))(psi)

#     return Multiply()([x, psi_up]) # Now psi has been upsampled.

# def attention_unet(img_rows, img_cols, nb_classes):
#     inputs = Input(shape=(img_rows, img_cols, 1))

#     # Encoding Path
#     c1 = Conv2D(64, kernel_size=3, padding='same', activation='relu')(inputs)
#     c1 = Conv2D(64, kernel_size=3, padding='same', activation='relu')(c1)
#     p1 = MaxPooling2D(pool_size=(2,2))(c1)

#     c2 = Conv2D(128, kernel_size=3, padding='same', activation='relu')(p1)
#     c2 = Conv2D(128, kernel_size=3, padding='same', activation='relu')(c2)
#     p2 = MaxPooling2D(pool_size=(2,2))(c2)

#     c3 = Conv2D(256, kernel_size=3, padding='same', activation='relu')(p2)
#     c3 = Conv2D(256, kernel_size=3, padding='same', activation='relu')(c3)
#     p3 = MaxPooling2D(pool_size=(2,2))(c3)

#     c4 = Conv2D(512, kernel_size=3, padding='same', activation='relu')(p3)
#     c4 = Conv2D(512, kernel_size=3, padding='same', activation='relu')(c4)
#     p4 = MaxPooling2D(pool_size=(2,2))(c4)

#     c5 = Conv2D(1024, kernel_size=3, padding='same', activation='relu')(p4)
#     c5 = Conv2D(1024, kernel_size=3, padding='same', activation='relu')(c5)

#     # Decoding Path with Attention Gates
#     u4 = UpSampling2D(size=(2,2))(c5)
#     a4 = attention_gate(c4, u4, 512)  # Attention Gate
#     # Ensure u4 has the same number of filters as a4 before concatenation
#     u4 = Conv2D(512, kernel_size=1, padding='same')(u4) #added

#         # Resize a4 or u4 to match shapes before concatenation
#     a4_shape = tf.keras.backend.int_shape(a4)
#     u4_shape = tf.keras.backend.int_shape(u4)

#     if a4_shape[1] != u4_shape[1] or a4_shape[2] != u4_shape[2]:
#         # Resize the larger tensor to match the smaller one
#         if a4_shape[1] > u4_shape[1] or a4_shape[2] > u4_shape[2]:
#             a4 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (u4_shape[1], u4_shape[2])))(a4)
#         else:
#             u4 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (a4_shape[1], a4_shape[2])))(u4)


#     m4 = Concatenate()([a4, u4])
#     d4 = Conv2D(512, kernel_size=3, padding='same', activation='relu')(m4)
#     d4 = Conv2D(512, kernel_size=3, padding='same', activation='relu')(d4)

#     u3 = UpSampling2D(size=(2,2))(d4)
#     a3 = attention_gate(c3, u3, 256)
#     # Ensure u3 has the same number of filters as a3 before concatenation
#     u3 = Conv2D(256, kernel_size=1, padding='same')(u3)  #added

#         # Resize a3 or u3 to match shapes before concatenation
#     a3_shape = tf.keras.backend.int_shape(a3)
#     u3_shape = tf.keras.backend.int_shape(u3)

#     if a3_shape[1] != u3_shape[1] or a3_shape[2] != u3_shape[2]:
#         # Resize the larger tensor to match the smaller one
#         if a3_shape[1] > u3_shape[1] or a3_shape[2] > u3_shape[2]:
#             a3 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (u3_shape[1], u3_shape[2])))(a3)
#         else:
#             u3 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (a3_shape[1], a3_shape[2])))(u3)


#     m3 = Concatenate()([a3, u3])
#     d3 = Conv2D(256, kernel_size=3, padding='same', activation='relu')(m3)
#     d3 = Conv2D(256, kernel_size=3, padding='same', activation='relu')(d3)

#     u2 = UpSampling2D(size=(2,2))(d3)
#     a2 = attention_gate(c2, u2, 128)
#     # Ensure u2 has the same number of filters as a2 before concatenation
#     u2 = Conv2D(128, kernel_size=1, padding='same')(u2) #added

#         # Resize a2 or u2 to match shapes before concatenation
#     a2_shape = tf.keras.backend.int_shape(a2)
#     u2_shape = tf.keras.backend.int_shape(u2)

#     if a2_shape[1] != u2_shape[1] or a2_shape[2] != u2_shape[2]:
#         # Resize the larger tensor to match the smaller one
#         if a2_shape[1] > u2_shape[1] or a2_shape[2] > u2_shape[2]:
#             a2 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (u2_shape[1], u2_shape[2])))(a2)
#         else:
#             u2 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (a2_shape[1], a2_shape[2])))(u2)


#     m2 = Concatenate()([a2, u2])
#     d2 = Conv2D(128, kernel_size=3, padding='same', activation='relu')(m2)
#     d2 = Conv2D(128, kernel_size=3, padding='same', activation='relu')(d2)

#     u1 = UpSampling2D(size=(2,2))(d2)
#     a1 = attention_gate(c1, u1, 64)
#     # Ensure u1 has the same number of filters as a1 before concatenation
#     u1 = Conv2D(64, kernel_size=1, padding='same')(u1) #added

#         # Resize a1 or u1 to match shapes before concatenation
#     a1_shape = tf.keras.backend.int_shape(a1)
#     u1_shape = tf.keras.backend.int_shape(u1)

#     if a1_shape[1] != u1_shape[1] or a1_shape[2] != u1_shape[2]:
#         # Resize the larger tensor to match the smaller one
#         if a1_shape[1] > u1_shape[1] or a1_shape[2] > u1_shape[2]:
#             a1 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (u1_shape[1], u1_shape[2])))(a1)
#         else:
#             u1 = tf.keras.layers.Lambda(lambda image: tf.image.resize(image, (a1_shape[1], a1_shape[2])))(u1)


#     m1 = Concatenate()([a1, u1])
#     d1 = Conv2D(64, kernel_size=3, padding='same', activation='relu')(m1)
#     d1 = Conv2D(64, kernel_size=3, padding='same', activation='relu')(d1)

#     outputs = Conv2D(nb_classes, kernel_size=1, activation='softmax')(d1)

#     outputs = Lambda(lambda image: tf.image.resize(image, (img_rows, img_cols)))(outputs)

#     model = Model(inputs, outputs)
#     return model

### custom loss function

In [17]:
import tensorflow.keras.backend as K
# def jacc_coeff_loss_tissue(y_true, y_pred):
#     loss = 0
#     for i in range(nb_classes):
#         layer_weight = 1.0
#         if i == 1:
#             layer_weight=2.0
#         loss += layer_weight * tf.reduce_sum(tf.minimum(y_true[:,:,:,i], y_pred[:,:,:,i])) / \
#                     tf.reduce_sum(tf.maximum(y_true[:,:,:,i], y_pred[:,:,:,i]))
#     loss = (1 - loss / nb_classes)
#     return loss
def vertical_penalty_loss(y_true, y_pred, penalty_weight=0.1):
    """
    Adds a vertical penalty loss to enforce ordering of class labels.

    Parameters:
    - y_true: Ground truth labels, one-hot encoded (not used directly in loss).
    - y_pred: Predicted labels, logits, or probabilities with shape (batch, height, width, classes).
    - penalty_weight: Weight of the vertical penalty term.

    Returns:
    - Penalty loss value.
    """
    # Compute difference along the vertical axis (height axis)
    diff = y_pred[:, 1:, :, :] - y_pred[:, :-1, :, :]

    # Penalize negative differences (ordering violation)
    penalty = tf.keras.backend.relu(-diff)

    # Compute the mean penalty and scale it
    penalty_loss = penalty_weight * tf.keras.backend.mean(tf.keras.backend.sum(penalty, axis=-1))

    return penalty_loss

# def dice_loss(y_true, y_pred):
#     smooth = 1e-6  # Small constant to prevent division by zero
#     y_true_f = K.flatten(y_true)  # Flatten ground truth mask
#     y_pred_f = K.flatten(y_pred)  # Flatten predicted mask
#     intersection = K.sum(y_true_f * y_pred_f)  # Compute overlap
#     return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
    
# Combined Loss Function
# def combined_loss(y_true, y_pred, alpha=0.5, beta=0.3, gamma=0.2):
#     cce = tf.keras.losses.CategoricalCrossentropy()(y_true, y_pred)
#     dice = dice_loss(y_true, y_pred)
#     penalty = vertical_penalty_loss(y_true, y_pred)

#     return alpha * cce + beta * dice + gamma * penalty  # Weighted sum

# def combined_loss(y_true, y_pred):
#     return dice_loss(y_true, y_pred) + vertical_penalty_loss(y_true, y_pred)

### Model checkpoints

In [18]:
img_rows=H; img_cols=W

#sgd_1=tf.keras.optimizers.legacy.SGD(lr=0.005, momentum=0.05, decay=0.01, nesterov=False)
Adam= tf.keras.optimizers.Adam(learning_rate=0.0005, beta_1=0.9, beta_2=0.999, decay=0.0, amsgrad=False)  ###............


model =  attention_unet(img_rows,img_cols,nb_classes)

#model.compile(optimizer = Adam, loss = 'categorical_crossentropy', metrics = ['accuracy']) # ....
model.compile(optimizer= Adam, loss=lambda y_true, y_pred: tf.keras.losses.categorical_crossentropy(y_true, y_pred) + vertical_penalty_loss(y_true, y_pred), metrics = ['accuracy'])
#model.compile(optimizer= Adam, loss=combined_loss, metrics=['accuracy'])
#model.compile(optimizer = Adam, loss=combined_loss, metrics=['accuracy'])
#model.compile(optimizer = Adam, loss = jacc_coeff_loss_tissue, metrics = ['accuracy'])



model.summary()

/home/suraj/Suraj_env/lib/python3.10/site-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
I0000 00:00:1753094768.307774  857962 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1525 MB memory:  -> device: 0, name: NVIDIA A10, pci bus id: 0000:ca:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200, 400,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 200, 400,  │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 200, 400,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 100, 200,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 100, 200,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 100, 200,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 50, 100,   │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 50, 100,   │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 50, 100,   │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 25, 50,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 25, 50,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 25, 50,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 12, 25,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 12, 25,    │  4,719,616 │ max_pooling2d_3[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 12, 25,    │  9,438,208 │ conv2d_8[0][0]    │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 24, 50,    │          0 │ conv2d_9[0][0]    │
│ (UpSampling2D)      │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 25, 50,    │          0 │ up_sampling2d[0]

 Total params: 32,425,612 (123.69 MB)

 Trainable params: 32,425,612 (123.69 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
#model.load_weights(imgpath+'/Model_checkpoints/best_weights_'+testname+'.hdf5')

best_checkpoint = ModelCheckpoint(
    filepath='./../../../../../mnt/DATA/sda1/Suraj/New directory' + '/Model_checkpoints/best_weights_' + testname + '.weights.h5',  # Use .weights.h5
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=True,
    mode='auto'
)

cur_checkpoint = ModelCheckpoint(
    filepath='./../../../../../mnt/DATA/sda1/Suraj/New directory' + '/Model_checkpoints/cur_weights_' + testname + '.weights.h5',  # Use .weights.h5
    monitor='val_loss',
    verbose=0,
    save_best_only=False,
    save_weights_only=True,
    mode='auto'
)

# Define the learning rate scheduler
lrScheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    cooldown=0,
    verbose=1,
    min_lr=1e-7
)

#model.load_weights(imgpath+'/Model_checkpoints/best_weights_'+testname+'.hdf5')


### Start training

In [20]:
epochs=500

train_mini_batch_sz =4
val_mini_batch_sz = 2
num_train_batches, num_val_batches = 32, 32

### Epoch


In [21]:
training_gen = DataGen(train_images, train_labels,train_mini_batch_sz,mode=True) # if flag is not true, then data augmentation will happen on validation also
val_gen = DataGen(val_images, val_labels, val_mini_batch_sz,mode=True) # if flag is not true, then data augmentation will happen on validation also

steps_epoch=20 #train_images.shape[0] // train_mini_batch_sz
val_steps=20 #val_images.shape[0] // val_mini_batch_sz

fit_history = model.fit(
    x=training_gen,  # Generator for training data
    steps_per_epoch=steps_epoch,
    epochs=epochs,
    validation_data=val_gen,  # Generator for validation data
    validation_steps=val_steps,
    verbose=1,
    callbacks=[best_checkpoint, cur_checkpoint]
)


Epoch 1/500


I0000 00:00:1753094773.818269  858123 service.cc:148] XLA service 0x7f7eb0001de0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753094773.818330  858123 service.cc:156]   StreamExecutor device (0): NVIDIA A10, Compute Capability 8.6
2025-07-21 10:46:14.000099: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1753094774.728318  858123 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-07-21 10:46:15.476587: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:306] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.39GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-07-21 10:46:18.232865: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:306] Allocator (GPU_0_bfc) ran out of memory tr

ResourceExhaustedError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code

  File "/home/suraj/Suraj_env/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/suraj/Suraj_env/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/suraj/Suraj_env/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/suraj/Suraj_env/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.10/asyncio/base_events.py", line 603, in run_forever

  File "/usr/lib/python3.10/asyncio/base_events.py", line 1909, in _run_once

  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run

  File "/home/suraj/Suraj_env/lib/python3.10/site-packages/ipykernel/kernel

### Print epoch history

In [ ]:
print(fit_history.history.keys())
plt.rcParams['figure.figsize'] = (20,5)

plt.subplot(1,2,1)
plt.title("Loss vs epoch")
plt.plot(fit_history.history["loss"], "r-*", label='train_loss')
plt.plot(fit_history.history["val_loss"], "b-*", label='test_loss')
plt.legend()
plt.ylim(0,3)

plt.subplot(1,2,2)
plt.title("Acc vs epoch")
plt.plot(fit_history.history["accuracy"], "r-*", label='train_acc')
plt.plot(fit_history.history["val_accuracy"], "b-*", label='test_acc')
plt.legend()
plt.ylim(0,1)

### Save model

In [ ]:
#model.save('./../../../../../mnt/DATA/sda1/Suraj/New directory'+'/Model_checkpoints/'+testname+'.h5') # Saving whole model (architecture + weights + optimizer state)
model.save('./../../../../../mnt/DATA/sda1/Suraj/New directory/Model_checkpoints/' + testname + '.h5')
# del model  # deletes the existing model
print('FINISHED!!!!!!!')

### Create a new model and load the best weight for testing

In [ ]:
# model2 = load_model('./Model_checkpoints/'+testname+'.keras')
#model.load_weights('./../../../../../mnt/DATA/sda1/Suraj/New directory'+'/Model_checkpoints/best_weights_'+testname+'.weights.h5')





# model = load_model('./../../../../../mnt/DATA/sda1/Suraj/New directory/Model_checkpoints/' + testname + '.h5', compile=False)
model.load_weights('./../../../../../mnt/DATA/sda1/Suraj/New directory' + '/Model_checkpoints/best_weights_' + testname + '.weights.h5')

print('model loaded successfully')

### Obtain Predictions

In [ ]:
print(np.shape(test_images), np.shape(test_labels))

In [ ]:
test_preds = model.predict(test_images) #obtain predictions for glaucoma

print(np.shape(test_preds))
print('Predictions obtained ....')

### Visualize the predictions

In [ ]:
def DiceCoeff(y_true, y_pred):
    dice = np.zeros(nb_classes)
    for i in range(nb_classes):
        y_pred111 = np.round(y_pred[:,:,i])
        ttt = np.sum(np.minimum(y_true[:,:,i], y_pred111)) / \
                        (np.sum(y_true[:,:,i])+np.sum(y_pred111))
        dice[i] = 2*ttt
    return np.round((np.sum(dice)/nb_classes),2)

In [ ]:
img_data= np.copy(test_images)
label_data= np.copy(test_labels)
label_data2= np.copy(test_preds)

dc_all=np.zeros(len(test_images))
for i in range(len(test_images)):
    dc = DiceCoeff(label_data[i],label_data2[i])
    dc_all[i]=dc

# print(dc_all)
print(np.mean(dc_all),np.std(dc_all))


plt.rcParams.update({'font.size': 22})
f1 = plt.figure(figsize=(6,6))
plt.boxplot(dc_all,patch_artist=False,showmeans=True,showfliers=True,widths=0.5)
plt.grid(True)
plt.ylabel('Dice coefficient')
plt.xlabel('Test dataset')
plt.xticks([])
# plt.ylim(0.75,1)
plt.subplots_adjust(right=0.5,left=0.2)
# plt.savefig('./SegmentationDiceCoefficient' +'.png',dpi=200)
plt.show()

In [ ]:
print(np.min(dc_all),np.argmin(dc_all))

In [ ]:
@interact(x=(0,len(img_data)-1,1)) #Function responsible for creating the slider. Usage format @interact(start,end,step size)
def h(x=0): #Starting number of the slider
    f = plt.figure(figsize=(12,6))
    plt.rcParams.update({'font.size': 12})
    f.subplots_adjust(left = 0.02,right = 0.75,wspace = 0.01,hspace = 0.3)

    f.add_subplot(2, 2, 1)
    plt.imshow(img_data[x,:,:,0], cmap='gray')
    plt.title("Test image")

    f.add_subplot(2, 2, 2)
    color_label= process_label(np.argmax(label_data[x], axis=2)+1)
    plt.imshow(color_label)
    plt.title("Manual segmentation")

    f.add_subplot(2, 2, 3)
    plt.imshow(img_data[x,:,:,0], cmap='gray')
    plt.imshow(color_label,alpha=0.3)
    plt.title("Test image with manual segmentation")

    f.add_subplot(2, 2, 4)
    color_label2= process_label(np.argmax(label_data2[x], axis=2)+1)
    plt.imshow(color_label2)
    plt.title("Network prediction")


    # plt.savefig('./'+str(x) +'.png',dpi=200)

    plt.show()